# Assignment 6: Risk Scores, Summaries, Error Handling, and Classes
##### This assignment is an extension of assignment 5.

In [1]:
"""Assignment 6: hive risk scores, hive summaries, error handling, and a Hive class."""
from datetime import date, datetime

##### Setup from Assignment 5
Starter data and cleaning functions carried over from Assignment 5 so this notebook runs on its own.

In [2]:
# Starter Data
RAW_INSPECTIONS = [
    {"hive_id": "H-01", "inspection_date": "2026-08-01", "temp_f": 84,
     "weight_lb": "142.5", "mites": 3, "queen_seen": True, "notes": "calm"},
    {"hive_id": "H-01", "inspection_date": "2026-08-15", "temp_f": "89",
     "weight_lb": "137.0", "mites": "8", "queen_seen": False, "notes": "queen not found"},
    {"hive_id": "H-01", "inspection_date": "2026-08-15", "temp_f": "89",
     "weight_lb": "137.0", "mites": "8", "queen_seen": False, "notes": "queen not found"},
    {"hive_id": "h-02", "inspection_date": "2026-08-02", "temp_f": "85F",
     "weight_lb": 155.2, "mites": 2, "queen_seen": "yes", "notes": None},
    {"hive_id": "H-02", "inspection_date": "08/16/2026", "temp_f": 91,
     "weight_lb": " 151.4 ", "mites": 5, "queen_seen": True, "notes": ""},
    {"hive_id": "H-02", "inspection_date": "2026-08-32", "temp_f": 92,
     "weight_lb": 149.8, "mites": 6, "queen_seen": False, "notes": "bad date"},
    {"hive_id": "H-03", "inspection_date": "2026-08-03", "temp_f": None,
     "weight_lb": 130.0, "mites": 0, "queen_seen": True, "notes": "cool morning"},
    {"hive_id": "H-03", "inspection_date": "2026-08-17", "temp_f": 96,
     "weight_lb": None, "mites": 9, "queen_seen": "no", "notes": "light hive"},
    {"hive_id": "H-04", "inspection_date": "2026-08-04", "temp_f": 79,
     "weight_lb": 160.0, "mites": -1, "queen_seen": True, "notes": "counter reset"},
    {"hive_id": "H-04", "inspection_date": "2026-08-18", "temp_f": 81,
     "weight_lb": 158.5, "mites": 4, "queen_seen": True, "notes": "calm"},
    {"hive_id": " H-05 ", "inspection_date": "2026-08-05", "temp_f": "72",
     "weight_lb": "bad", "mites": 1, "queen_seen": True, "notes": "scale failed"},
    {"hive_id": "H-05", "inspection_date": "2026-08-19", "temp_f": 74,
     "weight_lb": 144.2, "mites": 3, "queen_seen": True, "notes": "retest"},
    {"hive_id": "H-06", "inspection_date": "2026-08-06", "temp_f": 101,
     "weight_lb": 120.0, "mites": 12, "queen_seen": False, "notes": "agitated"},
    {"hive_id": "H-06", "inspection_date": "2026-08-20", "temp_f": 97,
     "weight_lb": 109.4, "mites": 14, "queen_seen": False, "notes": "low stores"},
]

In [3]:
# Assignment 5 functions: parse_date, to_bool, normalize_record, clean_records
def parse_date(raw_date):
    """Parse YYYY-MM-DD or MM/DD/YYYY into a date; raise ValueError if invalid."""
    try:
        if "-" in raw_date:
            return datetime.strptime(raw_date, "%Y-%m-%d").date()
        return datetime.strptime(raw_date, "%m/%d/%Y").date()
    except (TypeError, ValueError) as exc:
        raise ValueError(f"Invalid inspection_date: {raw_date}") from exc


def to_bool(value):
    """Convert "yes"/"no" to True/False; keep booleans; raise ValueError otherwise."""
    if value == "yes":
        return True
    if value == "no":
        return False
    if isinstance(value, bool):
        return value
    raise ValueError(f"Invalid queen_seen: {value}")


def normalize_record(record: dict) -> tuple[dict | None, str]:
    """Return (clean_record, "") if valid, or (None, reason) if rejected."""
    try:
        hive_id = str(record.get("hive_id", "")).strip().upper()
        if len(hive_id) != 4 or not hive_id.startswith("H-") or not hive_id[2:].isdigit():
            raise ValueError(f"Invalid hive_id: {hive_id}")

        inspection_date = parse_date(record.get("inspection_date"))

        temp_f = record.get("temp_f")
        if temp_f is not None:
            temp_f = float(str(temp_f).removesuffix("F"))
            if temp_f < -20 or temp_f > 130:
                raise ValueError(f"temp_f out of range: {temp_f}")

        try:
            weight_lb = float(record.get("weight_lb"))
        except (TypeError, ValueError):
            weight_lb = None
        if weight_lb is not None and weight_lb < 0:
            raise ValueError(f"weight_lb cannot be negative: {weight_lb}")

        mites = int(record.get("mites"))
        if mites < 0:
            raise ValueError(f"mites cannot be negative: {mites}")

        queen_seen = to_bool(record.get("queen_seen"))

        notes = record.get("notes")
        if notes is None:
            notes = ""

    except (TypeError, ValueError) as exc:
        return None, str(exc)

    clean = {
        "hive_id": hive_id,
        "inspection_date": inspection_date,
        "temp_f": temp_f,
        "weight_lb": weight_lb,
        "mites": mites,
        "queen_seen": queen_seen,
        "notes": notes,
    }
    return clean, ""


def clean_records(records: list[dict]) -> tuple[list[dict], list[dict]]:
    """Normalize every record, then remove duplicates by hive_id + inspection_date."""
    clean = []
    rejected = []
    seen = set()

    for index, record in enumerate(records):
        result, reason = normalize_record(record)
        if result is None:
            rejected.append({"source_index": index, "hive_id": record.get("hive_id"),
                             "reason": reason})
            continue

        key = (result["hive_id"], result["inspection_date"])
        if key in seen:
            rejected.append({"source_index": index, "hive_id": result["hive_id"],
                             "reason": "duplicate inspection"})
            continue

        seen.add(key)
        clean.append(result)

    return clean, rejected

In [4]:
# Clean the raw data (Assignment 5 pipeline)
CLEAN_RECORDS, REJECTED = clean_records(RAW_INSPECTIONS)
print(len(CLEAN_RECORDS), "clean |", len(REJECTED), "rejected")

11 clean | 3 rejected


##### 1. (10 points) Hive Risk Score with Pseudocode

Write pseudocode first, then implement calculate_risk(record, previous_weight). The function must return a tuple containing an integer score and a list of reason strings. Use named constants for all thresholds.

| Condition | Points |
|---|---|
| mites is 8 or greater | 3 |
| queen_seen is False | 2 |
| weight decreased by at least 10 pounds from the previous valid weight | 2 |
| temp_f is below 55 or above 95 | 1 |

No previous weight or a missing current weight earns no weight change points. Your pseudocode must state this explicitly. Add risk fields to copies of the clean records in chronological order within each hive. Include boundary tests for mites equal to 8, exactly 10 pounds of loss, and temperature equal to 95.

##### 1. Pseudocode

calculate_risk(record, previous_weight):
* score = 0, reasons = empty list
* if mites >= 8: score + 3, reason
* if queen not seen: score + 2, reason
* if NO previous weight or NO current weight: no weight points
    else if previous weight - current weight >= 10: score + 2, reason
* if temp missing: no temp points
    else if temp < 55 or temp > 95: score + 1, reason
* return (score, reasons)

add_risk_fields(clean records):
* for each hive: sort its records oldest to newest
* previous weight = none at the start of each hive
* for each record: copy it, add risk score + reasons to the copy
* if the record has a weight: previous weight = that weight

In [5]:
# Risk thresholds and points
MITE_THRESHOLD = 8
MITE_POINTS = 3
QUEEN_MISSING_POINTS = 2
WEIGHT_LOSS_THRESHOLD_LB = 10
WEIGHT_LOSS_POINTS = 2
TEMP_LOW_F = 55
TEMP_HIGH_F = 95
TEMP_POINTS = 1


def calculate_risk(record: dict, previous_weight: float | None) -> tuple[int, list[str]]:
    """Score one clean record against the risk table.

    Args:
        record: A clean inspection record.
        previous_weight: The hive's last valid weight, or None if there is none.

    Returns:
        (score, reasons): integer score and one reason string per condition met.

    Raises:
        ValueError: If the record is missing a field the score needs.
    """
    # Error handling 1 (see Q3): a missing field fails loudly instead of scoring 0
    try:
        mites = record["mites"]
        queen_seen = record["queen_seen"]
        weight = record["weight_lb"]
        temp = record["temp_f"]
    except KeyError as exc:
        raise ValueError(f"record is missing field {exc}") from exc

    score = 0
    reasons = []

    if mites >= MITE_THRESHOLD:
        score += MITE_POINTS
        reasons.append(f"mites {mites} >= {MITE_THRESHOLD}")

    if not queen_seen:
        score += QUEEN_MISSING_POINTS
        reasons.append("queen not seen")

    # No previous weight or no current weight: no weight points
    if previous_weight is not None and weight is not None:
        loss = round(previous_weight - weight, 1)  # weights are to 0.1 lb; avoids float error
        if loss >= WEIGHT_LOSS_THRESHOLD_LB:
            score += WEIGHT_LOSS_POINTS
            reasons.append(f"lost {loss} lb")

    # Missing temperature: no temp points
    if temp is not None and (temp < TEMP_LOW_F or temp > TEMP_HIGH_F):
        score += TEMP_POINTS
        reasons.append(f"temp {temp} outside {TEMP_LOW_F}-{TEMP_HIGH_F}")

    return score, reasons

In [6]:
def records_for_hive(records: list[dict], hive_id: str) -> list[dict]:
    """Return one hive's records sorted oldest to newest (input order is not trusted)."""
    matches = [record for record in records if record["hive_id"] == hive_id]
    return sorted(matches, key=lambda record: record["inspection_date"])


def add_risk_fields(records: list[dict]) -> tuple[list[dict], list[dict]]:
    """Add risk_score and risk_reasons to copies of the clean records.

    Records are scored in date order within each hive, so previous_weight is always
    that hive's last valid weight. The input records are not changed.

    Returns:
        (scored, skipped): scored copies, and any records that could not be scored.
    """
    scored = []
    skipped = []

    for hive_id in sorted({record["hive_id"] for record in records}):
        previous_weight = None  # no previous weight at the start of each hive

        for record in records_for_hive(records, hive_id):
            # Error handling 2 (see Q3): skip one bad record, keep scoring the rest
            try:
                score, reasons = calculate_risk(record, previous_weight)
            except (TypeError, ValueError) as exc:
                skipped.append({"hive_id": hive_id, "inspection_date": record["inspection_date"],
                                "reason": str(exc)})
                continue

            scored_record = dict(record)  # a copy, so the clean record stays unchanged
            scored_record["risk_score"] = score
            scored_record["risk_reasons"] = reasons
            scored.append(scored_record)

            if record["weight_lb"] is not None:
                previous_weight = record["weight_lb"]

    return scored, skipped

In [7]:
def make_record(**changes):
    """Build a zero-risk test record; keyword arguments override fields."""
    record = {"hive_id": "H-99", "inspection_date": date(2026, 9, 1), "temp_f": 75.0,
              "weight_lb": 150.0, "mites": 0, "queen_seen": True, "notes": ""}
    record.update(changes)
    return record


def test_risk_boundaries():
    """Boundary tests for each threshold, plus the missing-weight rules."""
    # Mites: 8 scores, 7 does not
    assert calculate_risk(make_record(mites=8), None) == (3, ["mites 8 >= 8"])
    assert calculate_risk(make_record(mites=7), None) == (0, [])

    # Weight: exactly 10 lb lost scores, 9.9 does not
    assert calculate_risk(make_record(weight_lb=140.0), 150.0) == (2, ["lost 10.0 lb"])
    assert calculate_risk(make_record(weight_lb=140.1), 150.0) == (0, [])
    # 128.2 - 118.2 is 9.99999... in floating point; rounding keeps it at 10
    assert calculate_risk(make_record(weight_lb=118.2), 128.2)[0] == 2

    # Weight: no previous weight, or a missing current weight, earns no points
    assert calculate_risk(make_record(weight_lb=100.0), None) == (0, [])
    assert calculate_risk(make_record(weight_lb=None), 150.0) == (0, [])

    # Temperature: 95 is not above 95; 95.1 is
    assert calculate_risk(make_record(temp_f=95.0), None) == (0, [])
    assert calculate_risk(make_record(temp_f=95.1), None)[0] == 1
    assert calculate_risk(make_record(temp_f=None), None) == (0, [])

    print("All risk boundary tests passed")


test_risk_boundaries()

All risk boundary tests passed


In [8]:
# Score the clean records, in chronological order within each hive
SCORED, SKIPPED = add_risk_fields(CLEAN_RECORDS)


def print_scored(records):
    """Print hive, date, score, and reasons for each scored record."""
    for record in records:
        print(record["hive_id"], record["inspection_date"], record["risk_score"],
              record["risk_reasons"])


print_scored(SCORED)
print("Skipped:", SKIPPED)
print("Clean records unchanged:", all("risk_score" not in r for r in CLEAN_RECORDS))

H-01 2026-08-01 0 []
H-01 2026-08-15 5 ['mites 8 >= 8', 'queen not seen']
H-02 2026-08-02 0 []
H-02 2026-08-16 0 []
H-03 2026-08-03 0 []
H-03 2026-08-17 6 ['mites 9 >= 8', 'queen not seen', 'temp 96.0 outside 55-95']
H-04 2026-08-18 0 []
H-05 2026-08-05 0 []
H-05 2026-08-19 0 []
H-06 2026-08-06 6 ['mites 12 >= 8', 'queen not seen', 'temp 101.0 outside 55-95']
H-06 2026-08-20 8 ['mites 14 >= 8', 'queen not seen', 'lost 10.6 lb', 'temp 97.0 outside 55-95']
Skipped: []
Clean records unchanged: True


##### 2. (10 points) Hive Summary Functions

Implement:

def summarize_hive(records: list[dict], hive_id: str) -> dict:

The returned dictionary must contain inspection_count, first_date, last_date, average_temp, latest_weight, total_mites, maximum_risk, and risk_label. Ignore missing numeric values when calculating averages. Set risk_label to low for scores 0 through 2, review for 3 through 5, and urgent for 6 or greater.

Use at least two helper functions rather than placing all logic in summarize_hive. Decide what the function returns when the hive has no records, document the decision, and test it. Do not assume the input is already sorted.

##### 2. Decisions
* **Empty hive:** returns the same keys, with `inspection_count` 0 and every other field `None`. Not 0, because 0 mites or a risk of 0 ("low") would look like a real, healthy reading.
* **latest_weight:** the most recent weight that is not missing (H-03's last inspection has no weight, so its latest weight is from 08-03).
* **Input:** the scored records from Question 1.

In [9]:
REVIEW_MIN_SCORE = 3
URGENT_MIN_SCORE = 6
SUMMARY_FIELDS = ["hive_id", "inspection_count", "first_date", "last_date", "average_temp",
                  "latest_weight", "total_mites", "maximum_risk", "risk_label"]


def check_hive_id(hive_id) -> None:
    """Raise ValueError unless hive_id is "H-" followed by two digits."""
    # Error handling 3 (see Q3): a typo is an error, not an empty hive
    if not (isinstance(hive_id, str) and len(hive_id) == 4
            and hive_id.startswith("H-") and hive_id[2:].isdigit()):
        raise ValueError(f"Invalid hive_id: {hive_id!r}")


def average_ignoring_missing(values: list) -> float | None:
    """Mean of the values that are not None; None if every value is missing."""
    present = [value for value in values if value is not None]
    if not present:
        return None
    return sum(present) / len(present)


def latest_known(values: list):
    """Last value that is not None in an oldest-to-newest list; None if all are missing."""
    for value in reversed(values):
        if value is not None:
            return value
    return None


def risk_label(score: int) -> str:
    """low for 0-2, review for 3-5, urgent for 6 or greater."""
    if score >= URGENT_MIN_SCORE:
        return "urgent"
    if score >= REVIEW_MIN_SCORE:
        return "review"
    return "low"


def summarize_hive(records: list[dict], hive_id: str) -> dict:
    """Summarize one hive's scored inspection records.

    Records can arrive in any order; they are sorted by date first.

    Returns:
        A dict with every key in SUMMARY_FIELDS. A hive with no records returns
        inspection_count 0 and None for every other field.

    Raises:
        ValueError: If hive_id is not "H-" followed by two digits.
    """
    check_hive_id(hive_id)
    hive = records_for_hive(records, hive_id)

    if not hive:
        summary = dict.fromkeys(SUMMARY_FIELDS)  # every field None
        summary.update(hive_id=hive_id, inspection_count=0)
        return summary

    maximum_risk = max(record["risk_score"] for record in hive)
    return {
        "hive_id": hive_id,
        "inspection_count": len(hive),
        "first_date": hive[0]["inspection_date"],
        "last_date": hive[-1]["inspection_date"],
        "average_temp": average_ignoring_missing([record["temp_f"] for record in hive]),
        "latest_weight": latest_known([record["weight_lb"] for record in hive]),
        "total_mites": sum(record["mites"] for record in hive),
        "maximum_risk": maximum_risk,
        "risk_label": risk_label(maximum_risk),
    }

In [10]:
def test_summaries():
    """Unsorted input, missing values, risk labels, and the empty hive."""
    unsorted = list(reversed(SCORED))  # newest first, on purpose

    h06 = summarize_hive(unsorted, "H-06")
    assert h06["first_date"] == date(2026, 8, 6)
    assert h06["last_date"] == date(2026, 8, 20)
    assert h06["maximum_risk"] == 8 and h06["risk_label"] == "urgent"

    h03 = summarize_hive(unsorted, "H-03")
    assert h03["average_temp"] == 96.0      # the missing temp is ignored
    assert h03["latest_weight"] == 130.0    # the missing latest weight is skipped

    assert [risk_label(s) for s in (0, 2, 3, 5, 6)] == ["low", "low", "review", "review", "urgent"]

    empty = summarize_hive(SCORED, "H-07")
    assert empty["inspection_count"] == 0
    assert all(empty[field] is None for field in SUMMARY_FIELDS[2:])

    print("All summary tests passed")


def print_summaries(records):
    """Print the summary for every hive in the records."""
    for hive_id in sorted({record["hive_id"] for record in records}):
        print(summarize_hive(records, hive_id))


test_summaries()
print_summaries(SCORED)

All summary tests passed
{'hive_id': 'H-01', 'inspection_count': 2, 'first_date': datetime.date(2026, 8, 1), 'last_date': datetime.date(2026, 8, 15), 'average_temp': 86.5, 'latest_weight': 137.0, 'total_mites': 11, 'maximum_risk': 5, 'risk_label': 'review'}
{'hive_id': 'H-02', 'inspection_count': 2, 'first_date': datetime.date(2026, 8, 2), 'last_date': datetime.date(2026, 8, 16), 'average_temp': 88.0, 'latest_weight': 151.4, 'total_mites': 7, 'maximum_risk': 0, 'risk_label': 'low'}
{'hive_id': 'H-03', 'inspection_count': 2, 'first_date': datetime.date(2026, 8, 3), 'last_date': datetime.date(2026, 8, 17), 'average_temp': 96.0, 'latest_weight': 130.0, 'total_mites': 9, 'maximum_risk': 6, 'risk_label': 'urgent'}
{'hive_id': 'H-04', 'inspection_count': 1, 'first_date': datetime.date(2026, 8, 18), 'last_date': datetime.date(2026, 8, 18), 'average_temp': 81.0, 'latest_weight': 158.5, 'total_mites': 4, 'maximum_risk': 0, 'risk_label': 'low'}
{'hive_id': 'H-05', 'inspection_count': 2, 'first_d

##### 3. (5 pts) Lint and Add Error Handling

Implement pylint with nbqa and get the linting score to 80% or higher. If you get to 90% you will get 1 pt of extra credit, and if you get to 100% you will get 2 pts of extra credit.

Add error handling where it makes sense and explain why it makes sense where you added it. There should be at least 3 instances of proper error handling.

##### 3. Linting

Run from the repo root with the course `.pylintrc`:

```
python -m nbqa pylint assignments/week6/homework_6_answers.ipynb --rcfile=.pylintrc
```

Result: **10.00/10**

##### 3. Error handling

| # | Where | What it does | Why it makes sense there |
|---|---|---|---|
| 1 | `calculate_risk` | A missing field raises `ValueError` naming the field | Without it, a broken record could score 0 and be labeled "low": a false all-clear. Better to fail loudly. |
| 2 | `add_risk_fields` | Catches the error for one record, logs it in `skipped`, keeps going | It processes the whole apiary. One bad record should not stop every other hive from being scored, and the skipped list keeps an audit trail (same idea as Assignment 5's rejected list). |
| 3 | `summarize_hive` (via `check_hive_id`) | A malformed hive_id raises `ValueError` | A typo like `"H6"` would otherwise return the empty-hive summary, which looks exactly like a real hive with no inspections. |

In [11]:
def demo_error_handling():
    """Trigger each of the three error-handling cases."""
    # 1. calculate_risk: a record missing a field
    broken = make_record()
    del broken["mites"]
    try:
        calculate_risk(broken, None)
    except ValueError as exc:
        print("1.", exc)

    # 2. add_risk_fields: one record with text where a number belongs
    bad = make_record(inspection_date=date(2026, 9, 2), mites="many")
    scored, skipped = add_risk_fields([make_record(), bad])
    print("2.", len(scored), "scored |", skipped)

    # 3. summarize_hive: a malformed hive_id
    try:
        summarize_hive(SCORED, "H6")
    except ValueError as exc:
        print("3.", exc)


demo_error_handling()

1. record is missing field 'mites'
2. 1 scored | [{'hive_id': 'H-99', 'inspection_date': datetime.date(2026, 9, 2), 'reason': "'>=' not supported between instances of 'str' and 'int'"}]
3. Invalid hive_id: 'H6'


##### 4. (5 pts) Implement a class in your code

Change some of the functions you created for this assignment or the previous one to include classes and properly use at least two of static methods, class methods, and instance methods. Explain why a class makes sense for how you're using it in the docstring or in markdown.

##### 4. Answer
`summarize_hive` rebuilt as a `Hive` class. Every summary helper works on the same thing: one hive's records, sorted. The class filters and sorts once, then keeps them.

* **Class method** `from_records`: builds a Hive from the whole apiary's list. It creates the object, so it needs `cls`.
* **Static method** `label`: a score's label depends only on the score, not on any hive.
* **Instance methods** `maximum_risk`, `summary`: they read this hive's records through `self`.

In [12]:
class Hive:
    """One hive and its scored inspection records.

    Why a class: summarize_hive and its helpers all start from the same data, one
    hive's records sorted by date. A Hive sorts once when it is created and keeps
    the records, so each method reads self.records instead of filtering again.
    """

    def __init__(self, hive_id: str, records: list[dict]):
        check_hive_id(hive_id)
        self.hive_id = hive_id
        self.records = sorted(records, key=lambda record: record["inspection_date"])

    @classmethod
    def from_records(cls, records: list[dict], hive_id: str) -> "Hive":
        """Build a Hive from the full apiary list, keeping only this hive's records."""
        return cls(hive_id, [record for record in records if record["hive_id"] == hive_id])

    @staticmethod
    def label(score: int) -> str:
        """low for 0-2, review for 3-5, urgent for 6 or greater."""
        if score >= URGENT_MIN_SCORE:
            return "urgent"
        if score >= REVIEW_MIN_SCORE:
            return "review"
        return "low"

    def maximum_risk(self) -> int | None:
        """Highest risk score for this hive; None if it has no records."""
        if not self.records:
            return None
        return max(record["risk_score"] for record in self.records)

    def summary(self) -> dict:
        """Same dictionary as summarize_hive, built from this hive's records."""
        if not self.records:
            summary = dict.fromkeys(SUMMARY_FIELDS)
            summary.update(hive_id=self.hive_id, inspection_count=0)
            return summary

        highest = self.maximum_risk()
        return {
            "hive_id": self.hive_id,
            "inspection_count": len(self.records),
            "first_date": self.records[0]["inspection_date"],
            "last_date": self.records[-1]["inspection_date"],
            "average_temp": average_ignoring_missing([r["temp_f"] for r in self.records]),
            "latest_weight": latest_known([r["weight_lb"] for r in self.records]),
            "total_mites": sum(record["mites"] for record in self.records),
            "maximum_risk": highest,
            "risk_label": Hive.label(highest),
        }

In [13]:
def test_hive_class():
    """The class must produce the same summary as summarize_hive, including empty hives."""
    for hive_id in ["H-01", "H-02", "H-03", "H-04", "H-05", "H-06", "H-07"]:
        assert Hive.from_records(SCORED, hive_id).summary() == summarize_hive(SCORED, hive_id)
    print("Hive class matches summarize_hive for every hive")
    print(Hive.from_records(SCORED, "H-06").summary())


test_hive_class()

Hive class matches summarize_hive for every hive
{'hive_id': 'H-06', 'inspection_count': 2, 'first_date': datetime.date(2026, 8, 6), 'last_date': datetime.date(2026, 8, 20), 'average_temp': 99.0, 'latest_weight': 109.4, 'total_mites': 26, 'maximum_risk': 8, 'risk_label': 'urgent'}
